# Exercice 05 - Stockage dans MinIO (Data Lake)

## Objectifs
- Configurer Spark pour se connecter a MinIO
- Lire et ecrire des donnees dans MinIO
- Organiser les donnees en Bronze / Silver / Gold
- Comprendre les chemins S3

---

## 1. Rappel : Architecture du Data Lake

```
+------------------------------------------------------------------+
|                         MINIO (S3)                               |
+------------------------------------------------------------------+
|                                                                  |
|  +----------------+  +----------------+  +----------------+      |
|  |    BRONZE      |  |    SILVER      |  |     GOLD       |      |
|  |                |  |                |  |                |      |
|  | Donnees brutes |  | Donnees        |  | Donnees        |      |
|  | telles que     |  | nettoyees      |  | agregees       |      |
|  | recues         |  | et typees      |  | pret pour      |      |
|  |                |  |                |  | l'analyse      |      |
|  | s3a://bronze/  |  | s3a://silver/  |  | s3a://gold/    |      |
|  +----------------+  +----------------+  +----------------+      |
|                                                                  |
+------------------------------------------------------------------+
```

## 2. Configuration de Spark pour MinIO

In [1]:
from pyspark.sql import SparkSession

# Arreter la session existante si elle existe (necessaire pour charger les JARs)
try:
    spark.stop()
except:
    pass

# Configuration de Spark pour se connecter a MinIO
# Les JARs AWS sont telecharges automatiquement au premier lancement
spark = SparkSession.builder \
    .appName("Spark MinIO") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

print("Spark configure pour MinIO !")
print(f"Version: {spark.version}")

Spark configure pour MinIO !
Version: 4.1.1


### Explication de la configuration

| Option | Description |
|--------|-------------|
| `fs.s3a.endpoint` | Adresse du serveur MinIO |
| `fs.s3a.access.key` | Identifiant de connexion |
| `fs.s3a.secret.key` | Mot de passe |
| `fs.s3a.path.style.access` | Necessaire pour MinIO |
| `fs.s3a.impl` | Implementation du systeme de fichiers S3 |

## 3. Creation des buckets

In [3]:
pip install minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 16.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [minio]todome]
Note: you may need to restart the kernel to use updated packages.


In [4]:
from minio import Minio

# Connexion a MinIO
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False
)

# Creer les buckets s'ils n'existent pas
for bucket in ["bronze", "silver", "gold"]:
    if not client.bucket_exists(bucket):
        client.make_bucket(bucket)
        print(f"Bucket '{bucket}' cree")
    else:
        print(f"Bucket '{bucket}' existe deja")

Bucket 'bronze' existe deja
Bucket 'silver' existe deja
Bucket 'gold' existe deja


## 4. Creer des donnees de test

In [5]:
# Creer un DataFrame de ventes
ventes = [
    ("2025-01-01", "Laptop", "Paris", 2, 999.99),
    ("2025-01-01", "Souris", "Lyon", 10, 29.99),
    ("2025-01-02", "Clavier", "Paris", 5, 79.99),
    ("2025-01-02", "Ecran", "Marseille", 3, 299.99),
    ("2025-01-03", "Laptop", "Lyon", 1, 999.99),
    ("2025-01-03", "Casque", "Paris", 8, 149.99)
]

colonnes = ["date", "produit", "ville", "quantite", "prix_unitaire"]

df_ventes = spark.createDataFrame(ventes, colonnes)
df_ventes.show()

+----------+-------+---------+--------+-------------+
|      date|produit|    ville|quantite|prix_unitaire|
+----------+-------+---------+--------+-------------+
|2025-01-01| Laptop|    Paris|       2|       999.99|
|2025-01-01| Souris|     Lyon|      10|        29.99|
|2025-01-02|Clavier|    Paris|       5|        79.99|
|2025-01-02|  Ecran|Marseille|       3|       299.99|
|2025-01-03| Laptop|     Lyon|       1|       999.99|
|2025-01-03| Casque|    Paris|       8|       149.99|
+----------+-------+---------+--------+-------------+



## 5. Ecriture dans Bronze

Les chemins S3 ont le format : `s3a://bucket/chemin/fichier`

In [6]:
# Ecrire les donnees brutes dans Bronze
chemin_bronze = "s3a://bronze/ventes/raw"

df_ventes.write \
    .mode("overwrite") \
    .parquet(chemin_bronze)

print(f"Donnees ecrites dans : {chemin_bronze}")

Donnees ecrites dans : s3a://bronze/ventes/raw


In [7]:
# Verifier en relisant
df_bronze = spark.read.parquet(chemin_bronze)
print("Lecture depuis Bronze :")
df_bronze.show()

Lecture depuis Bronze :
+----------+-------+---------+--------+-------------+
|      date|produit|    ville|quantite|prix_unitaire|
+----------+-------+---------+--------+-------------+
|2025-01-02|  Ecran|Marseille|       3|       299.99|
|2025-01-02|Clavier|    Paris|       5|        79.99|
|2025-01-01| Laptop|    Paris|       2|       999.99|
|2025-01-03| Casque|    Paris|       8|       149.99|
|2025-01-03| Laptop|     Lyon|       1|       999.99|
|2025-01-01| Souris|     Lyon|      10|        29.99|
+----------+-------+---------+--------+-------------+



## 6. Transformation vers Silver

En Silver, on nettoie et on enrichit les donnees.

In [8]:
from pyspark.sql.functions import col, to_date

# Transformation : ajouter le montant total
df_silver = df_bronze \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("montant_total", col("quantite") * col("prix_unitaire"))

df_silver.show()
df_silver.printSchema()

+----------+-------+---------+--------+-------------+-------------+
|      date|produit|    ville|quantite|prix_unitaire|montant_total|
+----------+-------+---------+--------+-------------+-------------+
|2025-01-02|  Ecran|Marseille|       3|       299.99|       899.97|
|2025-01-02|Clavier|    Paris|       5|        79.99|       399.95|
|2025-01-01| Laptop|    Paris|       2|       999.99|      1999.98|
|2025-01-03| Casque|    Paris|       8|       149.99|      1199.92|
|2025-01-03| Laptop|     Lyon|       1|       999.99|       999.99|
|2025-01-01| Souris|     Lyon|      10|        29.99|        299.9|
+----------+-------+---------+--------+-------------+-------------+

root
 |-- date: date (nullable = true)
 |-- produit: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- quantite: long (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- montant_total: double (nullable = true)



In [9]:
# Ecrire dans Silver
chemin_silver = "s3a://silver/ventes/enriched"

df_silver.write \
    .mode("overwrite") \
    .parquet(chemin_silver)

print(f"Donnees ecrites dans : {chemin_silver}")

Donnees ecrites dans : s3a://silver/ventes/enriched


## 7. Agregation vers Gold

En Gold, on prepare les donnees pour l'analyse.

In [10]:
from pyspark.sql.functions import sum, count, avg

# Lire depuis Silver
df_silver = spark.read.parquet(chemin_silver)

# Agregation par ville
df_gold_ville = df_silver.groupBy("ville").agg(
    count("*").alias("nb_ventes"),
    sum("quantite").alias("total_quantite"),
    sum("montant_total").alias("chiffre_affaires"),
    avg("montant_total").alias("panier_moyen")
)

df_gold_ville.show()

+---------+---------+--------------+------------------+-----------------+
|    ville|nb_ventes|total_quantite|  chiffre_affaires|     panier_moyen|
+---------+---------+--------------+------------------+-----------------+
|Marseille|        1|             3|            899.97|           899.97|
|    Paris|        3|            15|           3599.85|          1199.95|
|     Lyon|        2|            11|1299.8899999999999|649.9449999999999|
+---------+---------+--------------+------------------+-----------------+



In [11]:
# Ecrire dans Gold
chemin_gold = "s3a://gold/ventes/par_ville"

df_gold_ville.write \
    .mode("overwrite") \
    .parquet(chemin_gold)

print(f"Donnees ecrites dans : {chemin_gold}")

Donnees ecrites dans : s3a://gold/ventes/par_ville


## 8. Verification du Data Lake

Listons les objets dans chaque bucket.

In [12]:
# Lister les objets dans chaque bucket
for bucket in ["bronze", "silver", "gold"]:
    print(f"\n=== Bucket: {bucket} ===")
    objets = client.list_objects(bucket, recursive=True)
    for obj in objets:
        print(f"  {obj.object_name} ({obj.size} octets)")


=== Bucket: bronze ===
  demo/mon_profil.json (84 octets)
  demo/premier_fichier.json (107 octets)
  ventes/raw/_SUCCESS (0 octets)
  ventes/raw/part-00000-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (657 octets)
  ventes/raw/part-00001-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1571 octets)
  ventes/raw/part-00002-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1563 octets)
  ventes/raw/part-00003-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1578 octets)
  ventes/raw/part-00005-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1592 octets)
  ventes/raw/part-00006-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1564 octets)
  ventes/raw/part-00007-efc0bd7d-c887-4bd3-af25-786540e96b73-c000.snappy.parquet (1571 octets)

=== Bucket: silver ===
  ventes/enriched/_SUCCESS (0 octets)
  ventes/enriched/part-00000-0c4c2503-b109-4d20-a91f-b34622bce784-c000.snappy.parquet (1818 octets)
  ventes/enriched/part-00001-0c4c2503-b10

## 9. Schema du pipeline

```
DONNEES SOURCES                      DATA LAKE (MinIO)
---------------                      -----------------

  [DataFrame]                        +-------------------+
       |                             |      BRONZE       |
       |  Ingestion brute            |  s3a://bronze/    |
       +--------------------------->|  ventes/raw       |
                                     +--------+----------+
                                              |
                                              | Nettoyage
                                              | Enrichissement
                                              v
                                     +-------------------+
                                     |      SILVER       |
                                     |  s3a://silver/    |
                                     |  ventes/enriched  |
                                     +--------+----------+
                                              |
                                              | Agregation
                                              | Indicateurs
                                              v
                                     +-------------------+
                                     |       GOLD        |
                                     |  s3a://gold/      |
                                     |  ventes/par_ville |
                                     +-------------------+
                                              |
                                              v
                                        Reporting
                                        BI Tools
```

---

## Exercice

**Objectif** : Creer votre propre pipeline Bronze / Silver / Gold

**Consigne** :
1. Creez un DataFrame de clients (id, nom, email, pays, date_inscription)
2. Ecrivez-le dans `s3a://bronze/clients/raw`
3. Transformez : ajoutez une colonne `annee_inscription` extraite de la date
4. Ecrivez dans `s3a://silver/clients/enriched`
5. Agregez : comptez les clients par pays
6. Ecrivez dans `s3a://gold/clients/par_pays`

A vous de jouer :

In [ ]:
# Donnees clients
clients = [
    (1, "Alice Martin", "alice@email.com", "France", "2023-03-15"),
    (2, "Bob Johnson", "bob@email.com", "USA", "2023-06-22"),
    (3, "Charlie Dupont", "charlie@email.com", "France", "2024-01-10"),
    (4, "Diana Smith", "diana@email.com", "UK", "2023-11-05"),
    (5, "Eve Bernard", "eve@email.com", "France", "2024-02-28")
]

colonnes = ["id", "nom", "email", "pays", "date_inscription"]

# TODO: Creez le DataFrame
my_df_bronze= spark.createDataFrame(clients, colonnes)
my_df_bronze.show()
# TODO: Ecrivez dans Bronze
my_chemin_bronze= "s3a://bronze/clients/raw"
my_df_bronze.write\
            .mode('overwrite')\
            .parquet(my_chemin_bronze)


+---+--------------+-----------------+------+----------------+
| id|           nom|            email|  pays|date_inscription|
+---+--------------+-----------------+------+----------------+
|  1|  Alice Martin|  alice@email.com|France|      2023-03-15|
|  2|   Bob Johnson|    bob@email.com|   USA|      2023-06-22|
|  3|Charlie Dupont|charlie@email.com|France|      2024-01-10|
|  4|   Diana Smith|  diana@email.com|    UK|      2023-11-05|
|  5|   Eve Bernard|    eve@email.com|France|      2024-02-28|
+---+--------------+-----------------+------+----------------+



In [15]:
# TODO: Transformez et ecrivez dans Silver
# Indice : utilisez year(to_date(col("date_inscription"))) pour extraire l'annee

from pyspark.sql.functions import year, to_date, col
my_chemin_silver=  "s3a://silver/clients/enriched"
my_df_silver= my_df_bronze\
                .withColumn('annee_inscription',year(to_date(col("date_inscription"))) )
my_df_silver.write\
        .mode('overwrite')\
        .parquet(my_chemin_silver)

In [27]:
# TODO: Agregez et ecrivez dans Gold
from pyspark.sql.functions import year, to_date, col
my_chemin_gold=  "s3a://gold/clients/par_pays"
df_gold_pays = my_df_silver.groupBy("pays").agg(
    count("*").alias("nb_clients")
)

df_gold_pays.show()
df_gold_pays.write\
        .mode('overwrite')\
        .parquet(my_chemin_gold)

# to read from the medaillon 
print('inside the bronze brut version')
my_chemin_bronze= "s3a://bronze/clients/raw"
spark.read.parquet(my_chemin_bronze).show()
print('inside the silver enriched version')
spark.read.parquet(my_chemin_silver).show()


print('inside the gold aggrgated version')
spark.read.parquet(my_chemin_gold).show()

+------+----------+
|  pays|nb_clients|
+------+----------+
|France|         3|
|   USA|         1|
|    UK|         1|
+------+----------+

inside the bronze brut version
+---+--------------+-----------------+------+----------------+
| id|           nom|            email|  pays|date_inscription|
+---+--------------+-----------------+------+----------------+
|  3|Charlie Dupont|charlie@email.com|France|      2024-01-10|
|  1|  Alice Martin|  alice@email.com|France|      2023-03-15|
|  5|   Eve Bernard|    eve@email.com|France|      2024-02-28|
|  4|   Diana Smith|  diana@email.com|    UK|      2023-11-05|
|  2|   Bob Johnson|    bob@email.com|   USA|      2023-06-22|
+---+--------------+-----------------+------+----------------+

inside the silver enriched version
+---+--------------+-----------------+------+----------------+-----------------+
| id|           nom|            email|  pays|date_inscription|annee_inscription|
+---+--------------+-----------------+------+----------------+-

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **configurer Spark pour MinIO**
- Le format des chemins S3 : `s3a://bucket/chemin`
- Comment **lire et ecrire** dans MinIO avec Spark
- L'organisation **Bronze / Silver / Gold**

### Prochaine etape
Dans le prochain notebook, nous apprendrons a ingerer des donnees depuis **PostgreSQL**.